# Seoul Terrain and Viewpoint Exploration

Compare candidate regions, diagnostic shading/silhouettes/wireframes, and 640 × 900 mm panel pairs before choosing a production view. No brush paths or G-code are generated. Original notebooks and GeoTIFFs are not modified.

Install `python -m pip install -e ".[research,dev]"` and select that environment in VSCode. Set `DATASCAPE_DATA_DIR` or edit the local directory below. The full workflow renders 18 cameras plus one native-sampling check.

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display
from datascape.rendering.exploration import run_exploration

REPO_ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "pyproject.toml").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory, or set REPO_ROOT explicitly.")
DATA_DIR = Path(os.environ.get("DATASCAPE_DATA_DIR", str(REPO_ROOT / "data/raw")))
MASTER = DATA_DIR / "SeoulNorth_v02_MASTER_30m.tif"
CONFIG = REPO_ROOT / "configs/scenes/seoul_viewpoint_exploration.yaml"
OUTPUT = Path(os.environ.get("DATASCAPE_VIEW_OUTPUT", str(REPO_ROOT / "outputs/seoul_viewpoints_v01")))
BASE_REVISION = "282dab31bf9928918f79ba562859e32f3e29907b"
if not MASTER.is_file():
    raise FileNotFoundError("Set DATA_DIR to the folder containing the MASTER GeoTIFF.")

In [ ]:
report = run_exploration(MASTER, CONFIG, OUTPUT, base_revision=BASE_REVISION)
for candidate in report["candidates"]:
    print(candidate["candidate_id"], candidate["center_wgs84_lon_lat"], candidate["bounds_utm_m"], candidate["roi_size_m"])
for view in report["views"]:
    print(view["view_id"], view["azimuth_deg"], view["elevation_deg"], view["distance_m"], view["vertical_exaggeration"])
assert report["input_hash_unchanged"]

In [ ]:
for name in ["candidate_rois.png", "terrain_shortlist.png", "candidate_A_views.png", "candidate_B_views.png", "candidate_C_views.png", "camera_controls.png", "native_sampling_check.png"]:
    display(Image(filename=str(OUTPUT / name), width=1200))

## Review questions

- Which look direction gives useful ridge overlap and height?
- Does the portrait pair preserve the important peak and lowland transition?
- Does the 40 mm gutter divide a peak or valley?
- Is visible lower whitespace actually outside the cropped ROI? Hatching marks this area.
- Compare 1× and 1.8× vertical scale before treating steepness as natural relief.
- Review native 30 m sampling before interpreting facets as rock planes.

Choose the composition before implementing Taebaek-style curves, new ridge strokes, vegetation, or the calibrated machine backend.